In [16]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFacePipeline
from typing import TypedDict
from dotenv import load_dotenv
import os

In [17]:
load_dotenv()
hf_token = os.getenv("HUGGINGFACE_API_KEY")

In [18]:
model = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-0.5B-Instruct",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 150,
        "temperature": 0.7,
        "do_sample": True
    }
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 395.64it/s]


In [19]:
# create State
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [20]:
# create outline
def create_outline(state:BlogState)->BlogState:

    # fetch title
    title=state['title']

    # call LLM & generate outline
    prompt = f'Generate a detailed outline for a blog on this topic:{title}'
    outline = model.invoke(prompt)

    # update in state
    state['outline']=outline

    return state

In [21]:
# create blog
def create_blog(state:BlogState)->BlogState:
    # title
    title = state['title']
    outline = state['outline']

    # generate blog
    prompt = f'Generate a detailed blog on the title:{title} using the following outline: {outline}'
    content = model.invoke(prompt)

    # update state
    state['content']= content

    return state

In [22]:
# create graph
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog',END)

# compile
workflow = graph.compile()

In [23]:
intial_state = {'title':'raise of AI in india'}
final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'raise of AI in india', 'outline': 'Generate a detailed outline for a blog on this topic:raise of AI in india\n\nCertainly! Here\'s a detailed outline for your blog titled "Raising the Bar: AI in India" that covers various aspects:\n\n### 1. Introduction\n- **Introduction to AI in India**\n  - Overview of AI trends and developments in India\n  - Importance of AI in Indian economy and society\n\n### 2. The Rise of AI in India\n- **Historical Context**\n  - Brief history of AI development in India (e.g., early attempts, current state)\n  \n- **Current Trends**\n  - Key areas where AI is making significant progress (e.g., healthcare, transportation, finance)\n\n### 3. Benefits of AI in India\n- **Healthcare**: Automation of tasks such as patient care and diagnostics\n', 'content': 'Generate a detailed blog on the title:raise of AI in india using the following outline: Generate a detailed outline for a blog on this topic:raise of AI in india\n\nCertainly! Here\'s a detailed outli

In [24]:
print(final_state['outline'])

Generate a detailed outline for a blog on this topic:raise of AI in india

Certainly! Here's a detailed outline for your blog titled "Raising the Bar: AI in India" that covers various aspects:

### 1. Introduction
- **Introduction to AI in India**
  - Overview of AI trends and developments in India
  - Importance of AI in Indian economy and society

### 2. The Rise of AI in India
- **Historical Context**
  - Brief history of AI development in India (e.g., early attempts, current state)
  
- **Current Trends**
  - Key areas where AI is making significant progress (e.g., healthcare, transportation, finance)

### 3. Benefits of AI in India
- **Healthcare**: Automation of tasks such as patient care and diagnostics



In [25]:
print(final_state['content'])

Generate a detailed blog on the title:raise of AI in india using the following outline: Generate a detailed outline for a blog on this topic:raise of AI in india

Certainly! Here's a detailed outline for your blog titled "Raising the Bar: AI in India" that covers various aspects:

### 1. Introduction
- **Introduction to AI in India**
  - Overview of AI trends and developments in India
  - Importance of AI in Indian economy and society

### 2. The Rise of AI in India
- **Historical Context**
  - Brief history of AI development in India (e.g., early attempts, current state)
  
- **Current Trends**
  - Key areas where AI is making significant progress (e.g., healthcare, transportation, finance)

### 3. Benefits of AI in India
- **Healthcare**: Automation of tasks such as patient care and diagnostics
- **Transportation**: Autonomous vehicles, logistics, and supply chain management
- **Finance**: AI-driven fraud detection, personalized financial advice, and improved risk management
- **Educ